# Cifrado de Imágenes para Azure - Prueba de Concepto

Este notebook documenta el proceso de cifrado seguro de imágenes médicas que se subirán a Azure Storage y procesarán en Compute Instances.

## Arquitectura de Seguridad

| Componente | Servicio Azure | Función |
|------------|----------------|---------|
| Gestión de Llaves | **Azure Key Vault** | Almacena llaves de cifrado |
| Almacenamiento | **Azure Blob Storage** (Private Endpoint) | Guarda imágenes cifradas |
| Procesamiento | **ML Compute Instance** (VNet) | Descifra y procesa |
| Validación | **Hash SHA-256** | Garantiza integridad |

## Flujo de Seguridad

```
Hospital Local → Cifrado → Azure Storage → Compute Instance → Descifrado → Procesamiento
     ↓                ↓            ↓              ↓              ↓
  [Imagen]    [Fernet + Hash]  [.enc + hash]  [Key Vault]  [Validación]
```

## 1. Importación de Librerías

Necesitamos:
- **hashlib**: Para calcular hash SHA-256 y validar integridad
- **cryptography.fernet**: Cifrado simétrico seguro (AES-128 en modo CBC)
- **os**: Manejo de archivos y directorios

In [3]:
import os
import hashlib
from cryptography.fernet import Fernet

ModuleNotFoundError: No module named 'cryptography'

## 2. Configuración de la Llave de Cifrado

### Entorno Local (Prueba)
Generamos una llave temporal para simular el flujo.

### Entorno Azure (Producción)
La llave se obtendría de **Azure Key Vault**:

```python
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

credential = DefaultAzureCredential()
client = SecretClient(vault_url="https://mi-kv-fedml.vault.azure.net/", credential=credential)
llave = client.get_secret("encryption-key").value.encode()
cifrador = Fernet(llave)
```

### Consideraciones de Seguridad
- La llave NUNCA se almacena en código
- Se usa Managed Identity para autenticación
- Key Vault está en la misma VNet (Private Endpoint)

In [ ]:
# ENTORNO LOCAL: Generación de llave temporal
LLAVE_LOCAL = Fernet.generate_key()
cifrador = Fernet(LLAVE_LOCAL)

print(f"Llave generada (simulación local):")
print(f"Longitud: {len(LLAVE_LOCAL)} bytes")
print(f"Tipo: {type(cifrador)}")
print("\n⚠️ En producción, esta llave vendría de Azure Key Vault")

## 3. Función de Validación de Integridad

El hash SHA-256 garantiza que la imagen no se modifique durante:
- La transmisión a Azure
- El almacenamiento
- El proceso de descifrado

### ¿Por qué SHA-256?
- Genera un "fingerprint" único de 256 bits
- Cualquier cambio mínimo en la imagen produce un hash completamente diferente
- Es computacionalmente imposible generar dos archivos con el mismo hash

### Flujo de Validación
1. Hash original calculado antes de cifrar
2. Hash final calculado después de descifrar
3. Si coinciden → Integridad garantizada ✅

In [ ]:
def calcular_hash(ruta_archivo):
    """
    Calcula el hash SHA-256 para validar integridad.
    
    Args:
        ruta_archivo (str): Ruta al archivo a hashear
        
    Returns:
        str: Hash hexadecimal de 64 caracteres
    """
    sha256_hash = hashlib.sha256()
    
    # Lectura en bloques para archivos grandes (eficiente en memoria)
    with open(ruta_archivo, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    
    return sha256_hash.hexdigest()

## 4. Creación de Directorios de Trabajo

Simulamos la estructura de almacenamiento:

| Directorio | Representa | En Azure |
|------------|------------|----------|
| `imagenes_originales/` | Hospital local | On-premises |
| `servidor_simulado/` | Azure Storage | Blob Container (cifrado) |
| `servidor_simulado/recuperada_*` | Compute Instance | Descifrado temporal en VM |

In [ ]:
# Crear directorios si no existen
os.makedirs("imagenes_originales", exist_ok=True)
os.makedirs("servidor_simulado", exist_ok=True)

print("✅ Directorios creados:")
print("  - imagenes_originales/ (origen local)")
print("  - servidor_simulado/ (simula Azure Storage)")

## 5. Función Principal: Flujo de Cifrado Seguro

Esta función simula todo el ciclo de vida de una imagen médica desde el hospital hasta Azure.

### Fases del Proceso

#### FASE A: Cifrado y "Subida" a Azure
1. Leer imagen original
2. Calcular hash de integridad
3. Cifrar con Fernet (AES-128)
4. "Subir" a Azure Storage (simulado con archivo .enc)

#### FASE B: Descifrado en Compute Instance
1. Recuperar llave de Key Vault
2. Descargar imagen cifrada de Storage
3. Descifrar en memoria segura
4. Guardar temporalmente para procesamiento

#### FASE C: Validación de Integridad
1. Calcular hash de imagen descifrada
2. Comparar con hash original
3. Si coinciden → Procesar con modelo FL
4. Si difieren → RECHAZAR (posible ataque)

In [ ]:
def simular_flujo_seguro(nombre_imagen):
    """
    Simula el flujo completo de cifrado/descifrado con validación de integridad.
    
    Args:
        nombre_imagen (str): Nombre del archivo a procesar
    """
    ruta_origen = f"imagenes_originales/{nombre_imagen}"
    ruta_destino = f"servidor_simulado/{nombre_imagen}.enc"
    
    print(f"\n{'='*60}")
    print(f"INICIANDO FLUJO SEGURO: {nombre_imagen}")
    print(f"{'='*60}\n")
    
    # ==================== PASO A: CIFRADO ====================
    print("📤 PASO A: Cifrado y subida a Azure Storage")
    print("-" * 60)
    
    hash_original = calcular_hash(ruta_origen)
    print(f"   Hash original: {hash_original[:16]}...")
    
    with open(ruta_origen, "rb") as f:
        datos_encriptados = cifrador.encrypt(f.read())
    
    with open(ruta_destino, "wb") as f:
        f.write(datos_encriptados)
    
    tamano_original = os.path.getsize(ruta_origen)
    tamano_cifrado = os.path.getsize(ruta_destino)
    
    print(f"   Tamaño original: {tamano_original:,} bytes")
    print(f"   Tamaño cifrado: {tamano_cifrado:,} bytes")
    print(f"   Overhead: {tamano_cifrado - tamano_original} bytes")
    print(f"   ✅ Imagen cifrada y guardada\n")
    
    # ==================== PASO B: DESCIFRADO ====================
    print("📥 PASO B: Descifrado en Compute Instance (Azure)")
    print("-" * 60)
    
    with open(ruta_destino, "rb") as f:
        datos_leidos = f.read()
        datos_desencriptados = cifrador.decrypt(datos_leidos)
    
    ruta_final = f"servidor_simulado/recuperada_{nombre_imagen}"
    with open(ruta_final, "wb") as f:
        f.write(datos_desencriptados)
    
    print(f"   ✅ Imagen descifrada correctamente\n")
    
    # ==================== PASO C: VALIDACIÓN ====================
    print("🔍 PASO C: Validación de Integridad")
    print("-" * 60)
    
    hash_final = calcular_hash(ruta_final)
    print(f"   Hash original: {hash_original}")
    print(f"   Hash final:    {hash_final}")
    
    if hash_original == hash_final:
        print(f"\n   ✅ ÉXITO: Los hashes coinciden")
        print(f"   ✅ La imagen es idéntica tras el proceso")
        print(f"   ✅ Listo para procesamiento con modelo FL")
    else:
        print(f"\n   ❌ ERROR: La integridad de la imagen se perdió")
        print(f"   ❌ Posible corrupción o ataque")
        print(f"   ❌ RECHAZAR imagen")
    
    print(f"\n{'='*60}\n")

## 6. Prueba con Imagen de Ejemplo

Antes de ejecutar, asegúrate de tener una imagen llamada `test.jpg` en el directorio `imagenes_originales/`.

Si no tienes una, puedes:
1. Copiar cualquier imagen al directorio
2. Renombrarla a `test.jpg`
3. O ejecutar: `curl -o imagenes_originales/test.jpg https://via.placeholder.com/600`

In [ ]:
# Verificar si existe la imagen de prueba
if os.path.exists("imagenes_originales/test.jpg"):
    simular_flujo_seguro("test.jpg")
else:
    print("⚠️ No se encontró 'test.jpg'")
    print("\nPor favor:")
    print("1. Coloca una imagen en 'imagenes_originales/'")
    print("2. Renómbrala a 'test.jpg'")
    print("3. Ejecuta esta celda nuevamente")

## 7. Verificación de Archivos Generados

Revisemos los archivos creados durante el proceso.

In [ ]:
print("📁 Estructura de directorios generada:\n")

if os.path.exists("imagenes_originales"):
    print("imagenes_originales/")
    for archivo in os.listdir("imagenes_originales"):
        tamano = os.path.getsize(f"imagenes_originales/{archivo}")
        print(f"  └─ {archivo} ({tamano:,} bytes)")

if os.path.exists("servidor_simulado"):
    print("\nservidor_simulado/ (simula Azure Storage)")
    for archivo in os.listdir("servidor_simulado"):
        tamano = os.path.getsize(f"servidor_simulado/{archivo}")
        tipo = "cifrado" if archivo.endswith(".enc") else "descifrado"
        print(f"  └─ {archivo} ({tamano:,} bytes) - {tipo}")

## 7.1 Validación Estructural: SSIM (Structural Similarity Index)

En el ámbito médico (como en procesamientos con **NVFLARE**), además de validar el hash a nivel de bits, se verifica la similitud estructural de los pixeles tras recuperar la imagen (**SSIM**). 

Un valor SSIM de `1.0` garantiza que la imagen recuperada es perceptualmente y estructuralmente idéntica a la original, evitando el riesgo de pérdida de calidad o alteración que podría llevar a diagnósticos erróneos.

In [ ]:
# Necesitarás OpenCV y scikit-image. Descomenta la siguiente línea si necesitas instalarlos:
# !pip install opencv-python-headless scikit-image numpy

import cv2
from skimage.metrics import structural_similarity as ssim
import os

nombre_imagen = "test.jpg"
ruta_original = f"imagenes_originales/{nombre_imagen}"
ruta_recuperada = f"servidor_simulado/recuperada_{nombre_imagen}"

if os.path.exists(ruta_original) and os.path.exists(ruta_recuperada):
    # Leer imágenes en escala de grises para un cálculo SSIM más eficiente y estándar
    img_orig = cv2.imread(ruta_original, cv2.IMREAD_GRAYSCALE)
    img_recup = cv2.imread(ruta_recuperada, cv2.IMREAD_GRAYSCALE)
    
    if img_orig is not None and img_recup is not None:
        # Calcular SSIM
        score, diff = ssim(img_orig, img_recup, full=True)
        
        print(f"📊 Índice de Similitud Estructural (SSIM): {score:.6f}")
        print(f"   (Rango: -1 a 1, donde 1.0 significa idénticas)\n")
        
        if score >= 0.9999:
            print("✅ ÉXITO: Las imágenes son estructuralmente idénticas.")
            print("✅ La encriptación/desencriptación es segura para uso médico (Lossless).")
        else:
            print("⚠️ ADVERTENCIA: Se detectaron cambios estructurales en la imagen.")
    else:
        print("❌ Error: No se pudieron procesar las imágenes como matriz de píxeles.")
else:
    print("⚠️ Advertencia: No se encontraron las imágenes para comparar.")

## 8. Integración con Azure - Código Producción

A continuación, el código real que se usaría en Azure (no ejecutar aquí).

### Configuración Requerida en Azure

In [ ]:
# ============================================================
# CÓDIGO PARA PRODUCCIÓN EN AZURE
# ============================================================

"""
# 1. Instalación de dependencias en Compute Instance
pip install azure-identity azure-keyvault-secrets azure-storage-blob cryptography

# 2. Configuración de Managed Identity
- Asignar System Managed Identity al Compute Instance
- Otorgar permisos en Key Vault:
  * Key Vault Secrets User
- Otorgar permisos en Storage:
  * Storage Blob Data Contributor

# 3. Variables de entorno requeridas
KEY_VAULT_URL = "https://mi-kv-fedml.vault.azure.net/"
STORAGE_ACCOUNT = "stfedml"
CONTAINER_NAME = "encrypted-medical-images"
SECRET_NAME = "encryption-master-key"
"""

# ==================== CÓDIGO PRODUCCIÓN ====================

from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient
from azure.storage.blob import BlobServiceClient
from cryptography.fernet import Fernet
import hashlib
import os

class SecureImageProcessor:
    """Procesador seguro de imágenes médicas en Azure."""
    
    def __init__(self, key_vault_url, storage_account, container_name, secret_name):
        # Autenticación con Managed Identity
        self.credential = DefaultAzureCredential()
        
        # Cliente Key Vault
        kv_client = SecretClient(vault_url=key_vault_url, credential=self.credential)
        llave = kv_client.get_secret(secret_name).value.encode()
        self.cifrador = Fernet(llave)
        
        # Cliente Storage
        blob_url = f"https://{storage_account}.blob.core.windows.net"
        self.blob_service = BlobServiceClient(blob_url, credential=self.credential)
        self.container_client = self.blob_service.get_container_client(container_name)
    
    def calcular_hash(self, datos):
        """Calcula SHA-256 de datos en memoria."""
        return hashlib.sha256(datos).hexdigest()
    
    def cifrar_y_subir(self, ruta_local, nombre_blob):
        """Cifra imagen local y sube a Azure Storage."""
        with open(ruta_local, "rb") as f:
            datos_originales = f.read()
        
        hash_original = self.calcular_hash(datos_originales)
        datos_cifrados = self.cifrador.encrypt(datos_originales)
        
        # Subir a Blob Storage
        blob_client = self.container_client.get_blob_client(f"{nombre_blob}.enc")
        blob_client.upload_blob(datos_cifrados, overwrite=True)
        
        # Guardar hash como metadata
        blob_client.set_blob_metadata({"hash_sha256": hash_original})
        
        return hash_original
    
    def descargar_y_descifrar(self, nombre_blob):
        """Descarga de Storage, descifra y valida integridad."""
        blob_client = self.container_client.get_blob_client(f"{nombre_blob}.enc")
        
        # Descargar
        datos_cifrados = blob_client.download_blob().readall()
        metadata = blob_client.get_blob_properties().metadata
        hash_esperado = metadata["hash_sha256"]
        
        # Descifrar
        datos_descifrados = self.cifrador.decrypt(datos_cifrados)
        hash_actual = self.calcular_hash(datos_descifrados)
        
        # Validar
        if hash_esperado != hash_actual:
            raise ValueError("❌ Integridad comprometida - hash no coincide")
        
        return datos_descifrados

# Uso en Compute Instance
# processor = SecureImageProcessor(
#     key_vault_url=os.environ["KEY_VAULT_URL"],
#     storage_account=os.environ["STORAGE_ACCOUNT"],
#     container_name=os.environ["CONTAINER_NAME"],
#     secret_name=os.environ["SECRET_NAME"]
# )
# 
# # Cifrar desde hospital
# hash_orig = processor.cifrar_y_subir("mamografia.dcm", "hospital1/img001")
# 
# # Descifrar en Azure
# datos = processor.descargar_y_descifrar("hospital1/img001")

## 9. Consideraciones de Seguridad

### ✅ Implementado
| Control | Descripción |
|---------|-------------|
| Cifrado en tránsito | TLS 1.2+ (HTTPS) |
| Cifrado en reposo | Fernet (AES-128 CBC) |
| Gestión de llaves | Azure Key Vault con RBAC |
| Validación de integridad | SHA-256 hash |
| Aislamiento red | Private Endpoints + NSG |
| Autenticación | Managed Identity (sin credenciales) |

